# 02 · Cleaning and Exploratory Analysis

Input: `data/master_videos.csv` (11,196 videos) and `data/lttstore_skus_*.csv`, both written by `01_collect.ipynb`.

Output: `data/videos_clean.csv` plus the figures used in the presentation.

## What this notebook decides

Three cleaning decisions, each justified against the data rather than asserted:

1. **Drop unavailable records.** A handful of rows carry zero views and null duration — deleted or
   privated uploads that still appear in the playlist.
2. **Separate the WAN Show.** A weekly ~90-minute livestream podcast that has been sitting inside
   "long-form video" and dragging every median it touches.
3. **Do not reclassify short old uploads as Shorts.** Some pre-2021 videos run under a minute.
   Shorts did not exist then, so duration alone would misclassify them — which is exactly why
   `01_collect` took the Shorts label from YouTube's own `/shorts` tab instead of a duration rule.

In [ ]:
import pathlib, glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

CWD = pathlib.Path.cwd()
NB_DIR = next((p for p in [CWD, CWD/"notebook", *CWD.parents] if (p/"data").is_dir()), CWD)
DATA = NB_DIR / "data"
FIG  = NB_DIR / "figures"; FIG.mkdir(exist_ok=True)

raw = pd.read_csv(DATA/"master_videos.csv", parse_dates=["date"])
chan_subs = pd.read_csv(DATA/"channels.csv").set_index("handle").subs.to_dict()
sku = pd.read_csv(sorted(glob.glob(str(DATA/"lttstore_skus_*.csv")))[-1])

ORDER  = ["LinusTechTips", "Techquickie", "ShortCircuit", "TechLinked"]
COLORS = {"LinusTechTips":"#2E86DE", "Techquickie":"#E55039",
          "ShortCircuit":"#F6A609", "TechLinked":"#16A085"}
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False})

print(f"{len(raw):,} videos · {raw.date.min().date()} → {raw.date.max().date()}")
print(f"{sku.handle.nunique()} products · {len(sku)} variant rows")

## 1. Data quality audit

Look before deciding what to drop.

In [ ]:
print("Missing values")
miss = raw.isna().sum()
print(miss[miss > 0].to_string() if miss.any() else "  none")

print("\nSuspect records (zero views or null duration)")
bad = raw[(raw.views.fillna(0) == 0) | raw.duration_sec.isna()]
print(bad[["channel", "date", "title"]].to_string(index=False))
print(f"\n{len(bad)} rows, in two distinct failure modes:")
print(f"  {((raw.views.fillna(0)==0) & raw.duration_sec.isna()).sum()} missing both views and duration")
print(f"  {((raw.views.isna()) & raw.duration_sec.notna()).sum()} missing views but with a valid duration")
print("Neither is usable for a views-based analysis, so both are dropped in section 2 —")
print("but they are different problems and are counted separately there.")

In [ ]:
print("Duration distribution (minutes)")
print(raw.duration_min.describe([.25, .5, .75, .9, .99]).round(1).to_string())

bands = [(0,1,"< 1 min"), (1,3,"1–3 min"), (3,20,"3–20 min"),
         (20,60,"20–60 min"), (60,120,"1–2 h"), (120,np.inf,"> 2 h")]
print("\nBy band")
for lo, hi, lab in bands:
    n = raw.duration_min.between(lo, hi, inclusive="left").sum()
    print(f"  {lab:<10} {n:>6,}  ({n/len(raw):>5.1%})")

print("\nThe >1h tail is almost entirely one channel:")
print(raw[raw.duration_min > 60].channel.value_counts().to_string())

### 1a. What the long tail actually is

The WAN Show is Linus Media Group's weekly livestreamed podcast. It is a different product from a
scripted review — different length, different production cost, different viewing behaviour — and
it has been counted as "long-form video" throughout.

In [ ]:
is_wan = raw.title.str.contains("WAN Show", case=False, na=False)
ltt_long = raw[(raw.channel == "LinusTechTips") & (raw.kind == "long")]

summary = []
for label, sub in [("WAN Show", ltt_long[ltt_long.title.str.contains("WAN Show", case=False, na=False)]),
                   ("Everything else", ltt_long[~ltt_long.title.str.contains("WAN Show", case=False, na=False)])]:
    summary.append({"segment": label, "videos": len(sub),
                    "median_minutes": round(sub.duration_min.median(), 1),
                    "median_views": int(sub.views.median())})
print(pd.DataFrame(summary).to_string(index=False))

print(f"\nWAN Show spans {raw[is_wan].date.min().date()} -> {raw[is_wan].date.max().date()}")
print(f"and accounts for {is_wan.sum()/len(ltt_long):.1%} of the flagship's long-form catalog.")

In [ ]:
# Pre-2021 sub-minute uploads: legitimately short videos, NOT mislabelled Shorts.
tiny = raw[(raw.kind == "long") & (raw.duration_min <= 1)]
print(f"{len(tiny)} long-form videos run under a minute")
print(f"published {tiny.date.min().date()} → {tiny.date.max().date()}")
print(f"\nShorts did not exist before {raw[raw.kind=='short'].date.min().date()},")
print("so every one of these predates the format. A duration-based rule would have")
print("misclassified all of them — the /shorts tab label is the right source.")

## 2. Apply the cleaning rules

`format` becomes the analysis unit: `short`, `wan_show`, `long_video`.

In [ ]:
df = raw.copy()

# Rule 1 — drop unretrievable uploads.
# Note: zero views and null duration do NOT co-occur. Six rows have both;
# three more (2012 "Live Stream Hangout" uploads) have valid durations but null
# view counts. Both are unusable for a views-based analysis, so both are dropped,
# but they are different failure modes and are counted separately here.
n0 = len(df)
no_views = df.views.isna() | (df.views == 0)
no_dur   = df.duration_sec.isna()
print(f"Rule 1 · {(no_views & no_dur).sum()} rows missing both views and duration")
print(f"         {(no_views & ~no_dur).sum()} rows missing views only")
print(f"         {(~no_views & no_dur).sum()} rows missing duration only")
df = df[~(no_views | no_dur)].copy()
print(f"         dropped {n0-len(df)} → {len(df):,} remain")

# Rule 2 — split the livestream podcast out of long-form.
# The show predates its current name: it ran as "Live Show Archive" / "Live Stream
# Archive" from 2012. A title match alone also catches two short clips *about* the
# show, so the rule pairs the title with a duration floor.
WAN_TITLE = r"WAN Show|Live Show Archive|Live Stream Archive|Afterparty"
is_wan = (df.title.str.contains(WAN_TITLE, case=False, na=False)
          & (df.duration_sec >= 35*60)
          & (df.kind == "long"))
df["format"] = np.where(df.kind == "short", "short",
                        np.where(is_wan, "wan_show", "long_video"))
print(f"Rule 2 · {is_wan.sum()} episodes reclassified as wan_show "
      f"({df[df.format=='wan_show'].date.min().date()} → "
      f"{df[df.format=='wan_show'].date.max().date()})")
_title_only = df.title.str.contains(WAN_TITLE, case=False, na=False).sum()
print(f"         a title-only rule would have caught {_title_only}; the two extra are short")
print(f"         clips *about* the show (9.5 and 9.6 minutes). The 35-minute floor sits in a")
print(f"         natural gap — no episode runs between 10 and 38 minutes.")

# Rule 3 — Shorts label stays as YouTube assigned it (no duration override)
print(f"Rule 3 · Shorts label untouched ({(df.format=='short').sum()} videos)")

df["year"] = df.date.dt.year
df.to_csv(DATA/"videos_clean.csv", index=False)

print("\nFinal composition")
print(df.pivot_table(index="channel", columns="format", values="id",
                     aggfunc="count", fill_value=0).reindex(ORDER).to_string())
print(f"\nRemaining long_video entries over 60 minutes: "
      f"{((df.format=='long_video') & (df.duration_min>60)).sum()} "
      f"(marathons, build streams and similar one-offs, not a recurring format)")

## 3. The rolling baseline index

**An approach that was built, tested, and then demoted. Section 4b explains why.**

A video's view count is cumulative, so comparing raw counts across periods favours older videos.
The accrual curve cannot be recovered from cross-sectional data — age and era are perfectly
collinear, so a decline in recent performance is indistinguishable from views not yet accrued
(see `data/README.md` for the isotonic-regression attempt that established this).

The idea: divide each video by the median of the **k videos that preceded it on the same channel
and in the same format**. Because the comparison group sits adjacent in time, era is held
approximately constant by construction and no accrual estimate is needed. An index of 1.0 means a
video matched its recent neighbours; 2.0 means it doubled them.

The logic is sound and the index is useful — but section 4b shows it carries structural serial
correlation and therefore **cannot support significance testing**. It is kept here for ranking and
outlier detection within a format. The inferential claims in this project rest on raw view counts
over adjacent, non-overlapping periods instead.

In [ ]:
K = 9   # ViewStats' Outlier Score uses the same window

def add_rolling_index(d, k=K):
    d = d.sort_values("date").copy()
    # shift(1) so a video never contributes to its own baseline
    base = (d.groupby(["channel", "format"])["views"]
              .transform(lambda s: s.shift(1).rolling(k, min_periods=k).median()))
    d["baseline"] = base
    d["rbi"] = d.views / base
    return d

df = add_rolling_index(df)
cov = df.rbi.notna().mean()
print(f"Index computed for {df.rbi.notna().sum():,} of {len(df):,} videos ({cov:.1%})")
print(f"The {df.rbi.isna().sum()} without one are the first {K} in each channel/format series.")
print(f"\nMedian index by construction sits near 1.0 — actual: {df.rbi.median():.3f}")
df.to_csv(DATA/"videos_clean.csv", index=False)

## 4. Does the Shorts conclusion survive the rolling baseline?

Three progressively stricter views of the same question. If the answer moves, the earlier
version was measuring something other than what it claimed.

In [ ]:
cut = df[df.format == "short"].date.min()

def ratio_table(d, label, per_channel_window=False):
    """Shorts vs long-form median views.

    per_channel_window=True restricts BOTH arms to the interval in which that
    channel actually published Shorts. A single global start date is not enough:
    Techquickie stopped making Shorts in 2023 while its long-form ran to 2026, so
    a start-only cut compares 2021-23 Shorts against 2021-26 long-form.
    """
    rows = []
    for ch in ORDER:
        g = d[d.channel == ch]
        sh = g[g.format == "short"]
        lg = g[g.format == "long_video"]
        if per_channel_window and len(sh):
            lo, hi = sh.date.min(), sh.date.max()
            lg = lg[(lg.date >= lo) & (lg.date <= hi)]
        rows.append({"channel": ch,
                     "short": sh.views.median(), "long_video": lg.views.median(),
                     "n_short": len(sh), "n_long": len(lg),
                     "ratio": round(sh.views.median()/lg.views.median(), 2)})
    t = pd.DataFrame(rows).set_index("channel"); t.columns.name = label
    return t

print("A · raw medians, full history  (superseded — see the notes below)")
print(ratio_table(df, "A")[["short","long_video","ratio"]].to_string(), "\n")

print(f"B · single global cut at {cut.date()} — start aligned only")
print(ratio_table(df[df.date >= cut], "B")[["short","long_video","ratio"]].to_string(), "\n")

print("B2 · each channel restricted to its own Shorts window — start AND end aligned")
b2 = ratio_table(df, "B2", per_channel_window=True)
print(b2.to_string(), "\n")

print("Shorts publication window per channel:")
for ch in ORDER:
    s = df[(df.channel == ch) & (df.format == "short")]
    print(f"  {ch:<16} {s.date.min().date()} → {s.date.max().date()}  ({len(s)} Shorts)")

print("\nC · rolling baseline index (shown for completeness; see 4b for why it")
print("    cannot be compared across formats)")
same_era = df[df.date >= cut]
t = same_era.pivot_table(index="channel", columns="format", values="rbi", aggfunc="median").reindex(ORDER)
t["short_vs_long"] = (t["short"] / t["long_video"]).round(2)
print(t.round(3).to_string())

> **Which table to quote.**
>
> **B2.** A single global cut aligns only the start of the comparison. Techquickie's Shorts
> programme ran 2021-09-30 → 2023-08-13 (1.9 years) while its long-form continues to 2026, so
> table B compares under two years of Shorts against five years of long-form on that channel — the "1.45×" it
> reports is a window artefact, not an era-matched result.
>
> Even B2 is imperfect: within a shared window the two arms can still differ in mean age, and the
> accrual confound described in 4b applies to both tables. Treat every figure here as descriptive.
>
> Table A is retained only to show the size of the correction. It is the family of numbers that
> produced the discarded "2.29×" claim, computed then against the pre-cleaning long-form median of
> 652,204 rather than the post-WAN-split 783,446 that A now shows.

In [ ]:
# Robustness: does excluding the WAN Show change the answer? And does k matter?
print("Sensitivity of the same-era Shorts/long ratio\n")

with_wan = same_era.copy()
with_wan["format"] = np.where(with_wan.format == "wan_show", "long_video", with_wan.format)
print("  WAN Show folded back into long-form:")
print(ratio_table(with_wan, "")[["ratio"]].to_string(), "\n")
print("  WAN Show held separate (the cleaning decision):")
print(ratio_table(same_era, "")[["ratio"]].to_string(), "\n")

for k in (5, 9, 15, 25):
    tmp = add_rolling_index(df, k)
    t = tmp[tmp.date >= cut].pivot_table(index="channel", columns="format",
                                         values="rbi", aggfunc="median").reindex(ORDER)
    r = (t["short"]/t["long_video"]).round(2)
    print(f"  k={k:<3} " + "  ".join(f"{c[:12]}={r[c]}" for c in ORDER))

## 4b. Is the rolling baseline statistically defensible?

Two separate questions, often conflated:

1. **Is `k` justified?** Choosing 9 because another tool uses 9 is an appeal to authority, not a
   derivation. A defensible choice needs a criterion.
2. **Can the index carry inference?** Even a well-chosen `k` does not automatically license
   p-values. This turns out to be the binding constraint.

In [ ]:
from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.stats.diagnostic import acorr_ljungbox

# Criterion: a baseline that fully absorbs era effects leaves no correlation between the
# index and time. But a window that is too wide spans too long a period and lets era drift
# back in. That is a bias-variance tradeoff with an interior optimum.

rows = []
for k in [3, 5, 7, 9, 12, 15, 20, 25, 30, 40, 50]:
    d = add_rolling_index(df, k)
    d = d[np.isfinite(np.log(d.rbi.replace(0, np.nan)))].copy()
    d["lrbi"] = np.log(d.rbi)                    # ratios are right-skewed; work in logs
    cors, acf_fail, groups = [], 0, 0
    for _, g in d.groupby(["channel", "format"]):
        if len(g) < 60:
            continue
        groups += 1
        cors.append(abs(spearmanr(g.date.astype("int64"), g.lrbi).statistic))
        p = acorr_ljungbox(g.lrbi, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
        acf_fail += p < 0.05
    rows.append({"k": k, "coverage": len(d)/len(df),
                 "resid_time_corr": round(np.median(cors), 4),
                 "log_index_sd": round(d.lrbi.std(), 3),
                 "autocorr_fail_pct": round(acf_fail/max(groups,1), 2)})

diag = pd.DataFrame(rows).set_index("k")
print(diag.to_string())

**Reading the diagnostics.**

*Residual time correlation* is negligible at every window (0.013–0.054); it varies without direction between k≈7
and k≈20 and rises beyond that, broadly as the bias argument predicts. *Log-index dispersion* falls
from 0.813 at k=3 and flattens by k≈15 — the variance benefit of a wider window is exhausted there.

So **any k between roughly 9 and 15 is defensible, and the exact value does not matter** because
every diagnostic is flat across that range. That is a real justification. It is also a weaker
claim than "k=9 is optimal", which the data does not support.

**The binding constraint is the last column.** Ljung-Box rejects independence in 57–86% of
channel/format series at *every* k. This is not a tuning failure — it is structural. A video's own
views enter the baseline of the next k videos, so serial correlation is mechanically built in.

**Consequence: the index is a normalisation, not a residual with known sampling properties.**
Use it to rank videos and spot outliers within a format. Do **not** compute p-values on it.

---

### A confound the block bootstrap does *not* fix

The next section compares two adjacent two-year windows. Separating the windows controls
nothing about accrual — it *maximises* the age gap this notebook already declared unidentifiable.
Median age at scrape in those windows:

| channel | recent window | prior window | accrual advantage to "prior" |
|---|---|---|---|
| Linus Tech Tips | 381 d | 1,150 d | 3.0× |
| ShortCircuit | 345 d | 1,152 d | 3.3× |
| TechLinked | 365 d | 1,110 d | 3.0× |
| Techquickie | 636 d | 1,136 d | 1.8× |

A block bootstrap widens intervals for serial correlation and does nothing for this. **Every
absolute "decline" figure below is therefore an upper bound on the true decline, contaminated by
an unknown amount of unaccrued viewing.**

Two things bound the contamination without eliminating it. Six of twelve comparison channels post
ratios ≥ 0.95 over the identical windows, and two exceed 1.0 (computed in `03_analysis` on
long-form only, i.e. a 180-second duration floor; without that filter the count is 4 of 12) — impossible if accrual dominated,
since a genuinely flat channel would be pushed well below 1. Against that, LTT's own within-period
age gradient is monotone (median views 1.20M at 60–365 days, 2.55M at 1,095–1,460 days), and that
gradient is exactly the quantity that cannot be separated into accrual versus era.

**The only clean claim is the peer-relative one**, because peers face the same accrual bias in the
same calendar windows and it cancels in the comparison. That comparison lives in `03_analysis`.
Read what follows as descriptive, not as an identified effect.

In [ ]:
lv = df[df.format == "long_video"]
from statsmodels.tsa.stattools import acf

# --- Why a moving-block bootstrap ---------------------------------------------
# Section 4b showed the index carries serial correlation. Raw views do too: a good
# run of topics lifts several consecutive videos. Resampling individual videos
# assumes independence and therefore understates the interval. A moving-block
# bootstrap resamples contiguous runs instead, preserving the local structure.

def block_length(s, cap=30, verbose=False):
    """First lag where the ACF of log(views) falls inside the white-noise band,
    floored at n**(1/3).

    Returns (length, hit_cap). The recent windows carry a monotone trend, so their
    ACF often never enters the band and the search hits the cap — that is reported
    rather than hidden, because a capped value is not a measured correlation length.
    """
    x = np.log(s.values)
    m = min(cap, len(x)//4)
    a = acf(x, nlags=m, fft=True)
    ci = 1.96/np.sqrt(len(x))
    lag = next((i for i in range(1, m+1) if abs(a[i]) < ci), None)
    hit_cap = lag is None
    L = max(lag if lag else m, int(round(len(x)**(1/3))))
    return L, hit_cap

def _mbb(x, L, rng):
    n = len(x); nb = int(np.ceil(n/L))
    starts = rng.integers(0, n-L+1, nb)
    return np.concatenate([x[s:s+L] for s in starts])[:n]

def boot_median_ratio(a, b, blocks=True, n=5000, seed=42, symmetric=True):
    """CI for median(a)/median(b). Series must be time-ordered.

    seed is local, so any cell is reproducible on its own.
    symmetric=True uses one block length for both arms — asymmetric lengths would
    mean the two medians are resampled under different dependence assumptions.
    """
    rng = np.random.default_rng(seed)
    av, bv = a.dropna().values, b.dropna().values
    if not blocks:
        r = [np.median(rng.choice(av, len(av), True))/np.median(rng.choice(bv, len(bv), True))
             for _ in range(n)]
        return np.percentile(r, [2.5, 97.5]), None
    La, capa = block_length(a.dropna())
    Lb, capb = block_length(b.dropna())
    L = max(La, Lb) if symmetric else None
    r = [np.median(_mbb(av, L if symmetric else La, rng)) /
         np.median(_mbb(bv, L if symmetric else Lb, rng)) for _ in range(n)]
    return np.percentile(r, [2.5, 97.5]), (La, Lb, capa, capb, L)

end = df.date.max()
rec = lv[lv.date >= end - pd.Timedelta(days=730)].sort_values("date")
pri = lv[(lv.date >= end - pd.Timedelta(days=1460)) & (lv.date < end - pd.Timedelta(days=730))].sort_values("date")

print("Two-year change in per-video reach\n")
print(f"{'channel':<16}{'ratio':>7}{'naive 95%':>18}{'block 95%':>18}  L  verdict")
for ch in ORDER:
    a = rec[rec.channel == ch].views
    b = pri[pri.channel == ch].views
    (nl, nh), _   = boot_median_ratio(a, b, blocks=False)
    (bl, bh), inf = boot_median_ratio(a, b, blocks=True)
    La, Lb, capa, capb, L = inf
    flag = "*" if (capa or capb) else " "
    if bh < 1:   v = "declined"
    elif bl > 1: v = "grew"
    else:        v = "NOT distinguishable from no change"
    print(f"{ch:<16}{a.median()/b.median():>7.2f}  [{nl:.2f}, {nh:.2f}]  [{bl:.2f}, {bh:.2f}]  {L:>2}{flag} {v}")

print("\n* = the ACF never entered the white-noise band within 30 lags for at least one arm,")
print("  so the block length is the cap rather than a measured correlation length. The cause")
print("  is the monotone trend in the recent window, i.e. the accrual confound above.")
print("  Both arms use the same L; asymmetric lengths would resample the two medians under")
print("  different dependence assumptions.")

**What the block bootstrap changes.**

Three of the four channels still show a decline whose interval excludes 1.0. **TechLinked does
not**: its naive interval [0.87, 0.98] becomes [0.78, 1.06] once serial correlation is respected.

The correct claim is therefore **"three of four channels declined; TechLinked is not
distinguishable from no change"** — not "all four declined". Every interval in this project uses
the block version. The naive column is shown only to make the size of the correction visible.

In [ ]:
lv = df[df.format == "long_video"]

# The Techquickie blackout effect.
# Note on windows: a moving-block bootstrap needs enough observations to form blocks.
# At n=6 or n=10 the block length collapses to 2 and the interval is not trustworthy —
# those rows are shown to demonstrate exactly that instability, not to be quoted.
tq = lv[lv.channel == "Techquickie"].sort_values("date")
pos = tq.index.get_loc(tq.date.diff().idxmax())

print("Median views after the blackout relative to before, by window\n")
for w in (6, 10, 20, 30):
    pre  = tq.iloc[max(0, pos-w):pos].views
    post = tq.iloc[pos:pos+w].views
    if len(post) < w:
        print(f"  +/-{w:<3} only {len(post)} videos available after the gap — skipped")
        continue
    (lo, hi), _ = boot_median_ratio(post, pre, blocks=True)
    note = "too few observations for a reliable block bootstrap" if w < 20 else (
           "usable" if hi < 1 else "CI includes 1.0")
    print(f"  +/-{w:<3} ratio {post.median()/pre.median():.2f}  "
          f"CI [{lo:.2f}, {hi:.2f}]  n={len(pre)}/{len(post)}   {note}")

print("\nQuote the +/-20 window.")
print("The +/-6 interval looks tight only because it is degenerate: with n=6 and block")
print("length 2 there are five possible block starts, so the resample space holds ~125")
print("distinct outcomes and the same interval reappears on every seed. Its stability is")
print("an artefact of having too little data to vary, not evidence of precision.")

In [ ]:
# Direction robustness: within-format index of the comeback videos, across k
print("Within-format index of Techquickie's post-blackout videos (1.0 = matches neighbours)\n")
for k in (5, 9, 15, 25):
    d = add_rolling_index(df, k)
    d = d[(d.channel == "Techquickie") & (d.format == "long_video")].dropna(subset=["rbi"])
    print(f"  k={k:<3} median index = {d[d.date >= '2026-03-01'].rbi.median():.2f}")
print("\nBelow 1.0 at every k. The magnitude depends on k; the direction does not.")

## 5. Channel differentiation — the Plan B thesis

Are the four channels genuinely distinct products, or the same thing four times?

In [ ]:
lv = df[df.format == "long_video"]
recent = lv[lv.date >= df.date.max() - pd.Timedelta(days=730)]   # last 2 years

profile = pd.DataFrame({
    "videos_2y":       recent.groupby("channel").size(),
    # Weeks in which the channel actually published — a channel that went dark for
    # 15 months should not be described as publishing 0.5 videos a week throughout.
    "per_week_nominal": (recent.groupby("channel").size() / 104).round(2),
    "per_week_active":  recent.groupby("channel").apply(
        lambda g: round(len(g) / max(g.date.dt.to_period("W").nunique(), 1), 2),
        include_groups=False),
    "median_minutes":  recent.groupby("channel").duration_min.median().round(1),
    "median_views":    recent.groupby("channel").views.median().round(0),
    "views_per_minute":(recent.groupby("channel").views.median()
                        / recent.groupby("channel").duration_min.median()).round(0),
}).reindex(ORDER)
profile["engagement_pct"] = (recent.groupby("channel")
                               .apply(lambda g: (g.likes.sum()/g.views.sum())*100,
                                      include_groups=False).round(2))
print("Long-form profile, last 24 months\n")
print(profile.to_string())
print("\nper_week_nominal spreads output over all 104 weeks; per_week_active divides by")
print("weeks with at least one upload. Techquickie's gap between the two is the blackout.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

for ch in ORDER:
    d = recent[recent.channel == ch]
    ax[0].scatter(d.duration_min, d.views, s=6, alpha=.25, color=COLORS[ch], label=ch)
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("Duration (minutes, log)"); ax[0].set_ylabel("Views (log)")
ax[0].set_title("Each channel occupies a distinct length band", loc="left", weight="bold")
ax[0].legend(fontsize=8, markerscale=2)

pos = np.arange(len(ORDER))
ax[1].barh(pos, profile.per_week_nominal, color=[COLORS[c] for c in ORDER])
ax[1].set_yticks(pos); ax[1].set_yticklabels(ORDER); ax[1].invert_yaxis()
ax[1].set_xlabel("Long-form videos per week (nominal)")
ax[1].set_title("Publishing cadence, last 24 months", loc="left", weight="bold")
for i, v in enumerate(profile.per_week_nominal):
    ax[1].text(v + .05, i, f"{v}", va="center", fontsize=9)

plt.tight_layout(); plt.savefig(FIG/"channel_differentiation.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))
yearly = (df[df.format != "short"]
            .pivot_table(index="year", columns="channel", values="id", aggfunc="count")
            .reindex(columns=ORDER).fillna(0))
yearly = yearly[yearly.index <= df.date.max().year]
ax.stackplot(yearly.index, *[yearly[c] for c in ORDER],
             labels=ORDER, colors=[COLORS[c] for c in ORDER], alpha=.85)
ax.set_title("Output by channel (long-form + WAN Show)", loc="left", weight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Videos published")
ax.legend(loc="upper left", fontsize=8)
ax.text(0.99, 0.03, f"{df.date.max().year} is a partial year",
        transform=ax.transAxes, ha="right", fontsize=8, color="#666")
plt.tight_layout(); plt.savefig(FIG/"output_by_channel.png", bbox_inches="tight")
plt.show()

## 5b. Network health — the finding that changed the thesis

Section 5 shows the four channels are differentiated. This section asks whether they are *healthy*,
and the answer reframes the whole analysis.

In [ ]:
end = df.date.max()
recent_w = lv[lv.date >= end - pd.Timedelta(days=730)]
prior_w  = lv[(lv.date >= end - pd.Timedelta(days=1460)) & (lv.date < end - pd.Timedelta(days=730))]

trend = pd.DataFrame({
    "videos_prior":  prior_w.groupby("channel").size(),
    "videos_recent": recent_w.groupby("channel").size(),
    "median_prior":  prior_w.groupby("channel").views.median(),
    "median_recent": recent_w.groupby("channel").views.median(),
}).reindex(ORDER)
trend["output_change"] = (trend.videos_recent / trend.videos_prior).round(2)
trend["views_change"]  = (trend.median_recent / trend.median_prior).round(2)

print("Long-form: prior 24 months vs most recent 24 months\n")
print(trend.to_string())
print("\nPoint estimates fall on all four channels, but see section 4b: once serial")
print("correlation is accounted for, only three of the four intervals exclude 1.0,")
print("and all four are contaminated by the accrual gap documented there.")
print("\nNo channel's median rose while another's fell. That is consistent with an")
print("external shift affecting the network as a whole. It is not evidence against")
print("internal audience transfer either — that is compatible with all four medians")
print("falling if the network-wide level fell at the same time. It does not rule out")
print("cannibalisation by Shorts or by platforms outside this dataset — nothing here")
print("tests that.")

In [ ]:
# Techquickie went dark. Quantify it.
tq = lv[lv.channel == "Techquickie"].sort_values("date")
gaps = tq.date.diff()
i = gaps.idxmax()
pos = tq.index.get_loc(i)

print(f"Longest publishing gap on Techquickie: {gaps.max().days} days")
print(f"  last video before : {tq.iloc[pos-1].date.date()}")
print(f"  first video after : {tq.iloc[pos].date.date()}\n")

by_year = (lv[lv.channel=="Techquickie"]
             .groupby(lv[lv.channel=="Techquickie"].date.dt.year)
             .agg(videos=("id","count"), median_views=("views","median")))
print(by_year.tail(8).to_string())
print("\n2025 is absent from this table because the channel published nothing that year.")

# Effect size is reported in section 4b with a confidence interval.
# The narrow +/-6 window's interval is tight but degenerate — see the note printed below.
pre  = tq.iloc[max(0,pos-20):pos].views.median()
post = tq.iloc[pos:pos+20].views.median()
print(f"\nMedian views, 20 videos either side: {pre:,.0f} -> {post:,.0f} ({post/pre:.2f}x)")
print("See section 4b for the interval and for why the +/-6 figure is not reportable.")

In [ ]:
# Techquickie is the one channel where Shorts out-median long-form.
# Two readings are consistent with the numbers below and this dataset cannot
# separate them: Shorts doing unusually well, or long-form doing unusually badly.
tq_w = df[(df.channel == "Techquickie")]
sh_lo, sh_hi = tq_w[tq_w.format == "short"].date.agg(["min", "max"])
tq_win = tq_w[(tq_w.date >= sh_lo) & (tq_w.date <= sh_hi)]
tl_win = df[(df.channel == "TechLinked") & (df.date >= sh_lo) & (df.date <= sh_hi)]
subs = chan_subs   # read from channels.csv, not hardcoded

print(f"Within Techquickie's own Shorts window ({sh_lo.date()} → {sh_hi.date()}):\n")
print(f"  Techquickie Shorts     {tq_win[tq_win.format=='short'].views.median():>10,.0f}")
print(f"  Techquickie long-form  {tq_win[tq_win.format=='long_video'].views.median():>10,.0f}")
print(f"  TechLinked  long-form  {tl_win[tl_win.format=='long_video'].views.median():>10,.0f}"
      f"   ({subs['TechLinked']/subs['Techquickie']:.0%} of Techquickie's subscribers)")
print("\nTechquickie's long-form median sits below a channel with under half its")
print("subscriber count. Whether that reflects the blackout, a content change, or")
print("something else is not identified by anything in this notebook.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

for ch in ORDER:
    s = (lv[lv.channel == ch].set_index("date").views
           .resample("QE").median().rolling(2, min_periods=1).mean())
    s = s[s.index >= "2017-01-01"]
    ax[0].plot(s.index, s.values, color=COLORS[ch], lw=2, label=ch)
ax[0].set_yscale("log"); ax[0].set_ylabel("Median views per video (log)")
ax[0].set_title("Per-video reach, all four channels", loc="left", weight="bold")
ax[0].legend(fontsize=8)

q = (lv[lv.channel == "Techquickie"].set_index("date")
       .resample("QE").size())
q = q[q.index >= "2017-01-01"]
ax[1].bar(q.index, q.values, width=70, color=COLORS["Techquickie"], alpha=.85)
ax[1].axvspan(pd.Timestamp("2024-11-29"), pd.Timestamp("2026-03-03"),
              color="#000", alpha=.08)
ax[1].text(pd.Timestamp("2025-07-15"), q.max()*.75, "459-day\nblackout",
           ha="center", fontsize=9, color="#555")
ax[1].set_ylabel("Videos per quarter")
ax[1].set_title("Techquickie output", loc="left", weight="bold")

plt.tight_layout(); plt.savefig(FIG/"network_health.png", bbox_inches="tight")
plt.show()

## 6. The storefront

> Scope: products **currently on sale** only. Delisted items are absent from the endpoint, so no
> launch-cadence time series is attempted — that would systematically understate early expansion.
> What follows is cross-sectional structure, which the snapshot supports.

In [ ]:
by_type = (sku.groupby("product_type")
             .agg(products=("handle", "nunique"), variants=("sku", "size"),
                  median_price=("price", "median"), min_price=("price", "min"),
                  max_price=("price", "max"),
                  oos_pct=("available", lambda s: round((1-s.mean())*100, 1)))
             .sort_values("variants", ascending=False))
print(by_type.to_string())

# Product-level aggregates must be computed over ALL variant rows, then reduced.
# Reducing first (drop_duplicates) and aggregating after evaluates .any() on a
# single arbitrary variant and inflates the out-of-stock rate roughly threefold.
prod_stats = sku.groupby("handle").agg(
    any_available=("available", "any"),
    min_price=("price", "min"),
    median_price=("price", "median"))

print(f"\n{sku.handle.nunique()} products · {len(sku)} variant rows "
      f"({sku.sku.nunique()} distinct SKUs; 8 rows null, 13 excess duplicates)")
print(f"Price ${sku.price.min():.2f}–${sku.price.max():.2f}, median ${sku.price.median():.2f} "
      f"(across variants)")
print(f"Out of stock: {(~sku.available).mean():.1%} of variants, "
      f"{(~prod_stats.any_available).mean():.1%} of products entirely")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))

# Use the per-product median price, not the first variant's price.
pp = prod_stats.median_price
ax[0].hist(pp, bins=35, color="#2E86DE", alpha=.85)
ax[0].axvline(pp.median(), color="#E55039", ls="--", label=f"median ${pp.median():.2f}")
ax[0].set_xlabel("Product median price (USD)"); ax[0].set_ylabel("Products")
ax[0].set_title("Price architecture", loc="left", weight="bold"); ax[0].legend(fontsize=8)

top = by_type.head(8).sort_values("variants")
ax[1].barh(range(len(top)), top.variants, color="#16A085", alpha=.85)
ax[1].set_yticks(range(len(top))); ax[1].set_yticklabels(top.index, fontsize=8)
ax[1].set_xlabel("Variant count")
ax[1].set_title("SKU depth by category", loc="left", weight="bold")

plt.tight_layout(); plt.savefig(FIG/"store_structure.png", bbox_inches="tight")
plt.show()

## 7. Summary of what cleaning changed

In [ ]:
print(f"Records            {len(raw):,} → {len(df):,}")
print(f"WAN Show split out {(df.format=='wan_show').sum()} episodes")
print(f"Rolling index      {df.rbi.notna().sum():,} videos ({df.rbi.notna().mean():.1%})")
print(f"\nFlagship long-form median views")
b = raw[(raw.channel=='LinusTechTips') & (raw.kind=='long')].views.median()
a = df[(df.channel=='LinusTechTips') & (df.format=='long_video')].views.median()
print(f"  before cleaning  {b:>12,.0f}")
print(f"  after cleaning   {a:>12,.0f}   ({a/b:.2f}x)")
print(f"\nFigures written to {FIG}")
for f in sorted(FIG.glob('*.png')):
    print(f"  {f.name}")

---

# Findings

Everything below comes from medians and counts. No statistical test is involved, and none is
needed — these are descriptions of what the catalog contains, not claims about what caused it.

The constraints established earlier still apply: absolute cross-period changes are contaminated by
accrual, so nothing here compares one era against another. These are snapshots.

In [ ]:
# The two tables the findings rest on.

fmt = (df[df.channel == "LinusTechTips"]
         .groupby("format")
         .agg(videos=("id", "count"),
              median_minutes=("duration_min", "median"),
              median_views=("views", "median"))
         .reindex(["long_video", "wan_show", "short"]))
print("Table 1 — Linus Tech Tips by content format (full history)\n")
print(fmt.round(1).to_string())

end = df.date.max()
r = df[(df.format == "long_video") & (df.date >= end - pd.Timedelta(days=730))]

profile = pd.DataFrame({
    "subscribers":     pd.Series(chan_subs),
    "median_minutes":  r.groupby("channel").duration_min.median().round(1),
    "median_views":    r.groupby("channel").views.median().round(0),
    "views_per_minute": (r.groupby("channel").views.median()
                         / r.groupby("channel").duration_min.median()).round(0),
    "like_rate_pct":   r.groupby("channel").apply(
                          lambda g: g.likes.sum()/g.views.sum()*100, include_groups=False).round(2),
    "comment_rate_per_1k": r.groupby("channel").apply(
                          lambda g: g.comments.sum()/g.views.sum()*1000, include_groups=False).round(2),
}).reindex(ORDER)
profile["views_per_1k_subs"] = (profile.median_views/profile.subscribers*1000).round(0)

print("\n\nTable 2 — Long-form profile by channel, last 24 months\n")
print(profile.to_string())

## How to read these tables

**Table 1** splits the flagship channel's entire catalog by content format. Each row is a different
kind of thing the company makes, not a different time period. The two columns that matter together
are `median_minutes` and `median_views` — they give the cost and the return of one unit of each
format.

**Table 2** profiles the four channels on long-form video only, over the last two years. Every row
is a channel; every column is a different question about it. `views_per_minute` divides reach by
runtime and asks how much attention a minute of production buys. `views_per_1k_subs` divides reach
by subscriber count and asks how much of the accumulated audience actually shows up.

## 1. The median describes the machine; the maximum describes a lottery ticket

The flagship's single best-performing upload has 42 million views. Its typical upload has about
780,000. A mean would sit between them and describe neither.

The median is the right summary here because a content catalog is extremely right-skewed — a
handful of breakouts carry enormous numbers while the bulk of the catalog clusters far below. What
a business needs to plan around is **what a normal week produces**, and that is the median.

## 2. The WAN Show is not the same business as the videos

| format | median runtime | median views |
|---|---|---|
| regular long-form | 9.4 min | 783,446 |
| WAN Show | **91.2 min** | **441,650** |
| Shorts | 0.6 min | 1,493,240 |

Ten times the runtime for roughly half the reach. As an audience-acquisition product that is a bad
trade, which is strong evidence it is not one — nobody produces a ninety-minute livestream to
attract strangers. It reads as a retention and community product instead.

This is why the cleaning step in section 2 mattered beyond tidiness. Mixing 712 podcast episodes
into "long-form video" was not a data-quality slip; it averaged together two different business
functions and made every resulting median describe nothing in particular.

## 3. The four channels are four attention price points, not one product repeated

Median runtimes form a clean ladder: 18.2, 13.1, 8.8, 5.8 minutes. The network covers different
attention budgets, which looks like portfolio design rather than accident.

But the efficiency spread inside that portfolio is wide. A minute of flagship content buys roughly
89,000 views; a minute of ShortCircuit content buys about 23,000 — a fourfold difference, while
the two channels publish at a similar weekly cadence.

## 4. Subscriber counts badly mispredict who actually shows up

| channel | subscribers | median views | views per 1,000 subscribers |
|---|---|---|---|
| TechLinked | 1.98 M | 424,654 | **214** |
| ShortCircuit | 2.54 M | 296,627 | 117 |
| Linus Tech Tips | 16.9 M | 1,624,960 | 96 |
| Techquickie | **4.31 M** | **204,880** | **48** |

Techquickie holds 2.2 times TechLinked's subscribers and earns under half the views per video.
Per thousand subscribers the gap is 4.5-fold.

Subscribers are an accumulated stock; views are current reach. A divergence this large says a
substantial part of that stock is no longer an audience. **On this channel the subscriber count has
stopped functioning as a measure of anything.**

## 5. Engagement inverts the ranking

Like rate runs TechLinked 4.70%, Techquickie 4.60%, flagship 3.21%, ShortCircuit 2.84%. Comment
rate points the same way.

The channels with the smallest audiences have the most involved ones. The flagship reaches widest
and shallowest — consistent with a large share of algorithmically-delivered passers-by — while the
satellites reach a narrower group that behaves like an audience rather than traffic.

## What this adds up to

Linus Media Group is not one large channel with three small ones attached. It is a portfolio
covering different depths of attention: the flagship supplies scale, the satellites supply
engagement, the podcast supplies community.

Within that portfolio, **Techquickie is the leg that has stopped working** — the largest accumulated
audience, the weakest current reach, and a 459-day gap in publishing that section 5b documents.

Every statement above is descriptive. Why the portfolio looks like this, whether the pattern is
specific to this company, and what it implies commercially are questions this notebook does not
answer. They require a comparison group, which is what `03_analysis.ipynb` builds.

---

## Handover

Cleaned dataset: `data/videos_clean.csv`, with `format` and `rbi` columns added.

**What this notebook established**

- Cleaning rules, applied and justified against the data rather than asserted
- Four channels are genuinely differentiated by length, cadence and reach
- Per-video reach has fallen on three of the four channels over two years (TechLinked's
  interval includes 1.0 under the block bootstrap and is not established)
- The rolling baseline index works for within-format ranking but cannot carry inference

**What it deliberately does not do**

- No peer comparison. "LMG is declining" is uninterpretable without knowing whether the sector is
  declining too. That control group, and the question of whether the decline is company-specific,
  belongs to `03_analysis.ipynb`.
- No significance testing on the rolling index, for the reasons in 4b.

**Reporting rules carried forward**

- Quote table **B2** in section 4 for the Shorts result — B aligns only the start date, and
  never the full-history ratios in A
- Quote the +/-20 window for the Techquickie blackout effect, never +/-6
- Describe LMG's position within the peer distribution; do not claim statistical significance
  against peers (n=4 channels)